Import everything needed. 


In [44]:
import os 
from dotenv import load_dotenv 

#langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.agents import create_agent



load_dotenv()

True

In [7]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print(groq_key)
print(jina_key)  

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
jina_4397863590ba4eb0863d5df1a4fd4c4cNm1XiCR9_VtM4tmudTgZJf2N3VgI


In [9]:
### loading data 

DATA_FILE_PATH = os.path.join("data", "HR_policy.txt")

In [12]:
### data ingestion 

loader = TextLoader(DATA_FILE_PATH, encoding = "utf-8")

documents = loader.load()

print(documents)

[Document(metadata={'source': 'data\\HR_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDuring probati

In [ ]:
"""
## Langchain documents 
## langchin process everything in the form of documents 

Documents:
    
    page_content:
        is actual data
    Metadata:
        is information about the data
        
        
"""

In [13]:
len(documents)

1

In [16]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [20]:
print(documents[0].metadata)

{'source': 'data\\HR_policy.txt'}


In [26]:
### splitting the text 

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)


chunks = text_splitter.split_documents(documents)

In [29]:
print(chunks), len(chunks)

[Document(metadata={'source': 'data\\HR_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\HR_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\HR_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

(None, 9)

In [38]:
print(chunks[0].page_content)
print(chunks[1].page_content)
print(chunks[2].page_content)


COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.
2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.


In [51]:
## Embedding our data

embeddings_model = JinaEmbeddings(model_name = "jina-embeddings-v2-base-en")

# embeddings_model.model_name

In [52]:
## storing the data in vector db 

# embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en", jina_api_key=jina_key)
vector_store = FAISS.from_documents(chunks, embeddings_model)

print("CHUNKS ARE STORED", vector_store.index.ntotal)

CHUNKS ARE STORED 9


In [55]:
## finding the similiarity score 
test_query = "How many sick leaves employees get"

top_matches = vector_store.similarity_search(test_query, k=2)

for i,match in enumerate(top_matches, start=1):
    print(match.page_content)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


In [58]:
## LLM model 
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature = 0
)

In [66]:
response = llm.invoke("is learning rag is hard? replay in funny way in 3 line")
print(response.content)

Learning RAG?  
It’s like trying to teach a cat to do calculus—fun, confusing, and you’ll still get a “purr‑formance” report!  
Just remember: if the model starts asking for a “retrieval” of your sanity, you’re doing it right.


## AI Agents

#### no memory

In [ ]:
from langchain.agents import create_agent 



In [76]:
hr_assistance = create_agent(
    model = llm,
    tools = [],
    system_prompt = """
            You are friendly assistat. always ans from the hr_pilicy documents. if any case 
            you don't konw the ans just say I DON'T KNOW.
    """
     
)

In [78]:
result = hr_assistance.invoke({"messages": [{"role": "user", "content": "leave policy"}]})

In [87]:
result["messages"][-1].content


"I DON'T KNOW."

#### for all this we need tool for the use agent 

In [92]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

def search_hr_policy(question:str)-> str:
    """
    search hr policy document for information about the leaves, work form home,
    probation, notice,code of conduct and holidas. 
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks) 

In [93]:
hr_assistance = create_agent(
    model = llm,
    tools = [search_hr_policy],
    system_prompt = """
            You are friendly assistat. always ans from the hr_pilicy documents. if any case 
            you don't konw the ans just say I DON'T KNOW.
    """
     
)

In [95]:
result = hr_assistance.invoke({"messages": [{"role": "user", "content": "leave policy"}]})

In [100]:
print(result["messages"][-1].content)

**Leave Policy**

- **Annual Leave**: All full‑time employees receive **20 days of paid annual leave** per calendar year.  
- **Request Procedure**: Submit your leave request through the HR portal **at least 5 working days in advance**.  
- **Carry‑over**: Unused annual leave may be carried forward to the next year, but only up to **5 days**.  
- **Sick Leave**: Employees are entitled to **10 paid sick days** per year.  
- **Medical Certificate**: A medical certificate is required for sick leave that exceeds **2 consecutive days**.  

If you need more details or have specific questions, feel free to ask!


In [101]:
result = hr_assistance.invoke({"messages": [{"role": "user", "content": "how may holiday are there"}]})
print(result["messages"][-1].content)

According to the HR policy, the company observes **12 public holidays each year**.


In [103]:
result = hr_assistance.invoke({"messages": [{"role": "user", "content": "how make llm agents"}]})
print(result["messages"][-1].content)

I DON'T KNOW.


In [104]:
result = hr_assistance.invoke({"messages": [{"role": "user", "content": "sing song for me"}]})
print(result["messages"][-1].content)

I DON'T KNOW.


In [105]:
result = hr_assistance.invoke({"messages": [{"role": "user", "content": "what is the leave plicy"}]})
print(result["messages"][-1].content)

**Leave Policy**

- **Annual Leave**: All full‑time employees receive **20 paid days** of annual leave each calendar year.  
- **Request Procedure**: Submit a leave request through the HR portal **at least 5 working days in advance**.  
- **Carry‑over**: Unused annual leave may be carried forward to the next year, but only up to **5 days**.  
- **Sick Leave**: Employees are entitled to **10 paid sick days** per year.  
- **Medical Certificate**: Required for sick leave that exceeds **2 consecutive days**.  

If you need more details or have a specific situation, let me know!


In [106]:
result = hr_assistance.invoke({"messages": [{"role": "user", "content": "which are the polices are there give me brife of all this"}]})
print(result["messages"][-1].content)

**HR Policies (brief overview)**  

| Policy | Key Points |
|--------|------------|
| **Leave Policy** | • 20 days of paid annual leave per calendar year. <br>• Leave requests must be submitted via the HR portal at least 5 working days in advance. <br>• Unused annual leave can be carried forward up to 5 days. <br>• 10 paid sick days per year; a medical certificate is required for sick leave longer than 2 consecutive days. |
| **Probation Period** | • All new employees undergo a 3‑month probation period from the date of joining. <br>• During probation, employees are not eligible for paid leave but may take unpaid leave with manager approval. <br>• Performance is reviewed at the end of probation to confirm employment. |
| **Holidays** | • The company observes 12 public holidays each year (as per the official holiday calendar). <br>• Employees working on a public holiday are eligible for compensatory leave. |

**Policies not covered in the search results**

- Work‑From‑Home  
- Notice (te

In [108]:
result = hr_assistance.invoke({"messages": [{"role": "user", "content": "what is company name"}]})
print(result["messages"][-1].content)

The company name is **Acme Corp**.
